In [1]:
from repository_knowledge_assistant.ingestion.clone import RepositoryCloner
from repository_knowledge_assistant.ingestion.load import RepositoryLoader
from repository_knowledge_assistant.ingestion.parse import RepositoryParser
from repository_knowledge_assistant.ingestion.chunk import RepositoryChunker
from repository_knowledge_assistant.ingestion.embed import RepositoryEmbedder
from repository_knowledge_assistant.search.elasticsearch import Index
from repository_knowledge_assistant.search.retrieve import Retriever
from repository_knowledge_assistant.llm import LLM
from repository_knowledge_assistant.assistant import RAG

url = "https://github.com/pypa/sampleproject.git" 
repo = RepositoryCloner().get_repository(url)
docs = RepositoryLoader().load(repo)
parser = RepositoryParser()
chunker = RepositoryChunker()
chunks = []
for doc in docs:
    docs2 = parser.parse(doc)
    for doc2 in docs2:
        chunks.append(chunker.chunk(doc2))
chunks = sum(chunks, [])

embed = RepositoryEmbedder()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
embeddings = embed.embed_documents(chunks)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [3]:
index = Index(index_name="sample_project")
index.create_index()

In [4]:
index.index_documents(embeddings)

In [5]:
llm = LLM()
retriever = Retriever(index, embed)
rag = RAG(retriever, llm)

In [8]:
answer = rag.rag('which license', 'hybrid')
print(answer)

The license for the sampleproject is the MIT License, as specified in the pyproject.toml file:

```
"License :: OSI Approved :: MIT License",
```

and the actual license text is included in the LICENSE.txt file, which is a standard MIT License text.
